## Description


In [1]:
library(phyloseq)
library(NetCoMi)
library(igraph)
library(dplyr)

Loading required package: SpiecEasi




Attaching package: ‘igraph’


The following object is masked from ‘package:SpiecEasi’:

    make_graph


The following objects are masked from ‘package:stats’:

    decompose, spectrum


The following object is masked from ‘package:base’:

    union



Attaching package: ‘dplyr’


The following objects are masked from ‘package:igraph’:

    as_data_frame, groups, union


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
# 1. OTU table import
## whole sample prevalence 10% filtering genus level feature table

otu <- read.csv("data/2_preprocessed/prv10/prv10_network.tsv", sep="\t", row.names = 1, check.names = TRUE)



# row = genus / columns = Sample
otu_mat <- as.matrix(otu)
OTU <- otu_table(otu_mat, taxa_are_rows = TRUE)


# 2. Taxonomy name preprocess
## ASV ID supposed to be like "ASV1|d__Bacteria;p__Firmicutes;..." 
asv_ids <- rownames(otu_mat)

## export to taxonomy line
tax_strings <- sapply(strsplit(asv_ids, "\\|"), function(x) x[length(x)])

## separated by ";" 
tax_split <- strsplit(tax_strings, ";")

## Rank annotation(K, P, C, O, F, G)
tax_mat <- do.call(rbind, lapply(tax_split, function(x) {
  x <- trimws(x)
  length(x) <- 6
  return(x)
}))
colnames(tax_mat) <- c("Kingdom","Phylum","Class","Order","Family","Genus")
rownames(tax_mat) <- asv_ids

## prefix (d__, p__ ..etc) delete
tax_mat <- apply(tax_mat, 2, function(x) gsub("^[a-z]__", "", x))

TAX <- tax_table(as.matrix(tax_mat))


# 3. Import metadata
meta <- read.csv("data/mmi_fit_sepa.tsv",sep='\t', row.names = 1, check.names = FALSE) ## if want mmibased? row.names = 1


## If metadata have SampleID column, set columns names by rownames.

## sample_data conversion
SAM <- sample_data(meta)



# 4. OTU table and metadata align
## Sample name unification
colnames(otu_mat) <- gsub("-", "_", colnames(otu_mat))
rownames(meta)   <- gsub("-", "_", rownames(meta))

# select only intersection
common_samples <- intersect(colnames(otu_mat), rownames(meta))
otu_mat <- otu_mat[, common_samples]
meta <- meta[common_samples, ]

# make object
OTU <- otu_table(otu_mat, taxa_are_rows = TRUE)
SAM <- sample_data(meta)


# 5. make phyloseq object
ps <- phyloseq(OTU, TAX, SAM)

## check
ps

phyloseq-class experiment-level object
otu_table()   OTU Table:         [ 189 taxa and 143 samples ]
sample_data() Sample Data:       [ 143 samples by 15 sample variables ]
tax_table()   Taxonomy Table:    [ 189 taxa by 6 taxonomic ranks ]

## phyloseq separation by group(all control, mmi high, mmi low, pibd)

In [3]:
make_genus_phyloseq_for_netcomi <- function(ps_obj, group_values, group_var = "mmibased") {
  # sample metadata, mmibased
  meta <- as(sample_data(ps_obj), "data.frame")
  
  if (!group_var %in% colnames(meta)) {
    stop(sprintf("'%s' column not found in sample_data(ps_obj).", group_var))
  }
  
  # subset할 sample 선택
  keep_samples <- rownames(meta)[meta[[group_var]] %in% group_values]
  
  if (length(keep_samples) == 0) {
    stop("No samples matched group_values.")
  }
  
  ps_sub <- prune_samples(keep_samples, ps_obj)
  ps_sub <- prune_samples(sample_sums(ps_sub) > 0, ps_sub)
  
  # genus agglomeration
  ps_genus <- ps_sub
  
  taxtab <- as(tax_table(ps_genus), "matrix")
  
  genus   <- trimws(as.character(taxtab[, "Genus"]))
  family  <- trimws(as.character(taxtab[, "Family"]))
  order   <- trimws(as.character(taxtab[, "Order"]))
  class_  <- trimws(as.character(taxtab[, "Class"]))
  phylum  <- trimws(as.character(taxtab[, "Phylum"]))
  kingdom <- trimws(as.character(taxtab[, "Kingdom"]))
  
  bad <- function(x) is.na(x) | x == "" | x == "__"
  
  fallback_label <- ifelse(!bad(family), family,
                    ifelse(!bad(order),  order,
                    ifelse(!bad(class_), class_,
                    ifelse(!bad(phylum), phylum,
                    ifelse(!bad(kingdom), kingdom, "Unknown")))))
  
  fallback_rank <- ifelse(!bad(family), "F",
                   ifelse(!bad(order),  "O",
                   ifelse(!bad(class_), "C",
                   ifelse(!bad(phylum), "P",
                   ifelse(!bad(kingdom), "K", "U")))))
  
  base_name <- ifelse(
    !bad(genus),
    genus,
    paste0(fallback_label, "(", fallback_rank, ")")
  )
  
  new_names <- base_name
  idx_unclass <- bad(genus)
  
  if (any(idx_unclass)) {
    new_names[idx_unclass] <- ave(
      base_name[idx_unclass],
      base_name[idx_unclass],
      FUN = function(x) paste0(seq_along(x), "_", x)
    )
  }
  
  dup_idx <- duplicated(new_names) | duplicated(new_names, fromLast = TRUE)
  if (any(dup_idx)) {
    new_names[dup_idx] <- ave(
      new_names[dup_idx],
      new_names[dup_idx],
      FUN = function(x) paste0(x, "_", seq_along(x))
    )
  }
  
  taxa_names(ps_genus) <- new_names
  ps_genus
}

In [4]:
# MMI high
high_genus_renamed <- make_genus_phyloseq_for_netcomi(
  ps_obj = ps,
  group_values = "mmihigh"
)
taxtab_high <- as(tax_table(high_genus_renamed), "matrix")

# MMI low
low_genus_renamed <- make_genus_phyloseq_for_netcomi(
  ps_obj = ps,
  group_values = "mmilow"
)
taxtab_low <- as(tax_table(low_genus_renamed), "matrix")

# PIBD
pibd_genus_renamed <- make_genus_phyloseq_for_netcomi(
  ps_obj = ps,
  group_values = "PIBD"
)
taxtab_pibd <- as(tax_table(pibd_genus_renamed), "matrix")

In [5]:
length(unique(taxa_names(taxtab_high))) == ntaxa(taxtab_high)

logical(0)

In [6]:
high_genus_renamed

phyloseq-class experiment-level object
otu_table()   OTU Table:         [ 189 taxa and 51 samples ]
sample_data() Sample Data:       [ 51 samples by 15 sample variables ]
tax_table()   Taxonomy Table:    [ 189 taxa by 6 taxonomic ranks ]

In [7]:
low_genus_renamed

phyloseq-class experiment-level object
otu_table()   OTU Table:         [ 189 taxa and 43 samples ]
sample_data() Sample Data:       [ 43 samples by 15 sample variables ]
tax_table()   Taxonomy Table:    [ 189 taxa by 6 taxonomic ranks ]

In [8]:
pibd_genus_renamed

phyloseq-class experiment-level object
otu_table()   OTU Table:         [ 189 taxa and 49 samples ]
sample_data() Sample Data:       [ 49 samples by 15 sample variables ]
tax_table()   Taxonomy Table:    [ 189 taxa by 6 taxonomic ranks ]

## Save to phyloseq

In [ ]:
high_to_save <- list(high_genus_renamed = high_genus_renamed, taxtab_high = taxtab_high)
saveRDS(high_to_save, file="data/3_results/5_network_analysis/5_1_network_inference/high_phyloseq.rds")

low_to_save <- list(low_genus_renamed = low_genus_renamed, taxtab_low = taxtab_low)
saveRDS(low_to_save, file="data/3_results/5_network_analysis/5_1_network_inference/low_phyloseq.rds")

pibd_to_save <- list(pibd_genus_renamed = pibd_genus_renamed, taxtab_pibd = taxtab_pibd)
saveRDS(pibd_to_save, file="data/3_results/5_network_analysis/5_1_network_inference/pibd_phyloseq.rds")

## Relative abundance export

In [9]:
# 1. OTU table import
## whole sample prevalence 10% filtering genus level feature table

otu <- read.csv("data/2_preprocessed/ab_rel.csv", sep=",", row.names = 1, check.names = TRUE)

# row = genus / columns = Sample
otu_mat <- as.matrix(otu)
OTU <- otu_table(otu_mat, taxa_are_rows = TRUE)


# 2. Taxonomy name preprocess
## ASV ID supposed to be like "ASV1|d__Bacteria;p__Firmicutes;..." 
asv_ids <- rownames(otu_mat)

## export to taxonomy line
tax_strings <- sapply(strsplit(asv_ids, "\\|"), function(x) x[length(x)])

## separated by ";" 
tax_split <- strsplit(tax_strings, ";")

## Rank annotation(K, P, C, O, F, G)
tax_mat <- do.call(rbind, lapply(tax_split, function(x) {
  x <- trimws(x)
  length(x) <- 6
  return(x)
}))

colnames(tax_mat) <- c("Kingdom","Phylum","Class","Order","Family","Genus")
rownames(tax_mat) <- asv_ids

## prefix (d__, p__ ..etc) delete
tax_mat <- apply(tax_mat, 2, function(x) gsub("^[a-z]__", "", x))

TAX <- tax_table(as.matrix(tax_mat))


# 3. Import metadata
meta <- read.csv("data/mmi_fit_sepa.tsv",sep='\t', row.names = 1, check.names = FALSE) ## if want mmibased? row.names = 1


## If metadata have SampleID column, set columns names by rownames.

## sample_data conversion
SAM <- sample_data(meta)



# 4. OTU table and metadata align
## Sample name unification
colnames(otu_mat) <- gsub("-", "_", colnames(otu_mat))
rownames(meta)   <- gsub("-", "_", rownames(meta))

# select only intersection
common_samples <- intersect(colnames(otu_mat), rownames(meta))
otu_mat <- otu_mat[, common_samples]
meta <- meta[common_samples, ]

# make object
OTU <- otu_table(otu_mat, taxa_are_rows = TRUE)
SAM <- sample_data(meta)


# 5. make phyloseq object
ps_ra <- phyloseq(OTU, TAX, SAM)

## check
ps_ra

phyloseq-class experiment-level object
otu_table()   OTU Table:         [ 189 taxa and 143 samples ]
sample_data() Sample Data:       [ 143 samples by 15 sample variables ]
tax_table()   Taxonomy Table:    [ 189 taxa by 6 taxonomic ranks ]

In [10]:
get_group_mean_ra_from_ra_ps <- function(ps_obj, group_values, group_var = "mmibased") {
  ## mmibased
  ps_genus <- make_genus_phyloseq_for_netcomi(
    ps_obj = ps_obj,
    group_values = group_values,
    group_var = group_var
  )
  
  otu_mat <- as(otu_table(ps_genus), "matrix")
  if (!taxa_are_rows(ps_genus)) {
    otu_mat <- t(otu_mat)
  }
  
  mean_ra <- rowMeans(otu_mat, na.rm = TRUE)
  
  out <- data.frame(
    Feature = taxa_names(ps_genus),
    MeanRelativeAbundance = as.numeric(mean_ra),
    stringsAsFactors = FALSE
  )
  
  out <- out[order(out$MeanRelativeAbundance, decreasing = TRUE), ]
  rownames(out) <- NULL
  out
}

In [11]:
mean_ra_high <- get_group_mean_ra_from_ra_ps(ps_ra, "mmihigh")
mean_ra_low  <- get_group_mean_ra_from_ra_ps(ps_ra, "mmilow")
mean_ra_pibd <- get_group_mean_ra_from_ra_ps(
  ps_ra,
  c("PIBD")
)

In [12]:
sum(mean_ra_high$MeanRelativeAbundance)
sum(mean_ra_low$MeanRelativeAbundance)
sum(mean_ra_pibd$MeanRelativeAbundance)

[1] 1

[1] 1

[1] 1

In [ ]:
write.csv(
    mean_ra_high,
    "data/3_results/5_network_analysis/5_2_netcomi_export/high/high_rel_abundance.csv",
    row.names = FALSE
)

write.csv(
    mean_ra_low,
    "data/3_results/5_network_analysis/5_2_netcomi_export/low/low_rel_abundance.csv",
    row.names = FALSE
)

write.csv(
    mean_ra_pibd,
    "data/3_results/5_network_analysis/5_2_netcomi_export/pibd/pibd_rel_abundance.csv",
    row.names = FALSE
)